In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

# Global variables
drawing = False  # True if the mouse is pressed
points = []  # List to store points for freehand drawing
#label_options = ["apple", "banana", "orange", "grape", "pear"]  # Predefined label options
label_options = os.listdir('/home/natsu/MechAI/Dataset')
img = None  # Global variable for the current image
img_copy = None  # Global variable for the image copy
selected_label = None  # Global variable to store the selected label

def draw_freehand(event, x, y, flags, param):
    global drawing, points, img, img_copy

    # Start drawing on left mouse button press
    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        points = [(x, y)]  # Start new path
        img_copy = img.copy()  # Copy of the image to reset as needed

    # Add points to path on mouse movement
    elif event == cv2.EVENT_MOUSEMOVE:
        if drawing:
            img_copy = img.copy()  # Reset to original image for a fresh redraw
            points.append((x, y))
            for i in range(1, len(points)):
                cv2.line(img_copy, points[i - 1], points[i], (0, 255, 0), 2)  # Draw line segments
            
            # Show label options on the image
            display_label_options(img_copy)  # Display options during drawing
            cv2.imshow("Image", img_copy)  # Show updated drawing in real-time

    # Stop drawing on left mouse button release and close shape
    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        points.append((x, y))
        cv2.line(img_copy, points[0], points[-1], (0, 255, 0), 2)  # Connect end to start
        display_label_options(img_copy)  # Display options after drawing
        cv2.imshow("Image", img_copy)  # Show the final closed shape

def display_label_options(image):
    global selected_label
    height, width = image.shape[:2]
    
    # Set background for label options (on the side of the image)
    label_area_width = 300  # Set width for the label options area
    font = cv2.FONT_HERSHEY_SIMPLEX
    y_offset = 30  # Initial vertical offset for the first label option
    
    # Add label options text to the image itself
    for i, label in enumerate(label_options):
        color = (255, 255, 255)  # White text for unselected labels
        if selected_label == label:
            color = (0, 255, 0)  # Green color for selected label
        cv2.putText(image, f"{i+1}: {label}", (width - label_area_width + 10, y_offset), font, 1, color, 2, cv2.LINE_AA)
        y_offset += 40  # Adjust vertical position for the next label

def crop_and_label_object(img_input, save_dir):
    global points, img, img_copy, selected_label
    img = img_input  # Assign the input image to the global img variable
    img_copy = img.copy()  # Initialize the image copy
    selected_label = None  # Reset selected label for each new image

    # Reset points for each new image
    points.clear()

    # Show image and set mouse callback for drawing
    cv2.imshow("Image", img_copy)
    cv2.setMouseCallback("Image", draw_freehand)

    # Display the label options panel on the image itself
    display_label_options(img_copy)

    # Wait until the user selects a label before or after drawing starts
    while True:
        key = cv2.waitKey(1) & 0xFF  # Check for a key press, process continuously
        if ord('1') <= key <= ord('5'):  # Check for valid number keys (1-5)
            selected_label = label_options[key - ord('1')]  # Select label based on key press
            print(f"Label '{selected_label}' selected.")
            break
        if key == ord('q'):  # Press 'q' to quit the image view
            print("Quitting...")
            break

    cv2.destroyAllWindows()

    if selected_label is None:
        print("No label selected, skipping this image.")
        return  # Skip if no label was selected

    # Create a mask with probable background
    mask = np.full(img.shape[:2], cv2.GC_PR_BGD, dtype=np.uint8)
    if len(points) > 2:
        # Create a polygon from points and set it to probable foreground
        points_array = np.array(points, dtype=np.int32)
        cv2.fillPoly(mask, [points_array], cv2.GC_PR_FGD)

        # Use GrabCut algorithm to refine the selection
        bgd_model = np.zeros((1, 65), np.float64)
        fgd_model = np.zeros((1, 65), np.float64)
        rect = cv2.boundingRect(points_array)  # Get a bounding box for the GrabCut
        cv2.grabCut(img, mask, rect, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_MASK)

        # Create a binary mask where the sure and probable foreground is set to 1
        mask2 = np.where((mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD), 1, 0).astype('uint8')
        result = img * mask2[:, :, np.newaxis]  # Apply mask to the image

        # Crop the result to the bounding box
        x, y, w, h = cv2.boundingRect(points_array)
        cropped_img = result[y:y+h, x:x+w]

        # Show cropped image
        plt.imshow(cv2.cvtColor(cropped_img, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.show()

        # Save the cropped image with the selected label
        save_path = f"{save_dir}/{selected_label}.png"
        cv2.imwrite(save_path, cropped_img)
        print(f"Cropped image saved as: {save_path}")
    else:
        print("Shape not drawn. Please draw a complete shape next time.")

def process_images_in_folder(image_folder, save_dir):
    # Get all image files in the specified folder
    image_files = glob.glob(os.path.join(image_folder, "*.jpg")) + \
                  glob.glob(os.path.join(image_folder, "*.png")) + \
                  glob.glob(os.path.join(image_folder, "*.jpeg"))

    for image_path in image_files:
        print(f"Processing image: {image_path}")
        img = cv2.imread(image_path)
        if img is not None:
            crop_and_label_object(img, save_dir)
        else:
            print(f"Failed to load image: {image_path}")

# Example usage
# Replace '/path/to/images' and '/path/to/save_dir' with actual paths
process_images_in_folder('/home/natsu/MechAI/Dataset/disk_rotor', '/home/natsu/MechAI/Dataset/disk_rotor_2')


Processing image: /home/natsu/MechAI/Dataset/disk_rotor/disk_rotor_132.png


QObject::moveToThread: Current thread (0x2bcbfd0) is not the object's thread (0x30e2ea0).
Cannot move to target thread (0x2bcbfd0)

QObject::moveToThread: Current thread (0x2bcbfd0) is not the object's thread (0x30e2ea0).
Cannot move to target thread (0x2bcbfd0)

QObject::moveToThread: Current thread (0x2bcbfd0) is not the object's thread (0x30e2ea0).
Cannot move to target thread (0x2bcbfd0)

QObject::moveToThread: Current thread (0x2bcbfd0) is not the object's thread (0x30e2ea0).
Cannot move to target thread (0x2bcbfd0)

QObject::moveToThread: Current thread (0x2bcbfd0) is not the object's thread (0x30e2ea0).
Cannot move to target thread (0x2bcbfd0)

QObject::moveToThread: Current thread (0x2bcbfd0) is not the object's thread (0x30e2ea0).
Cannot move to target thread (0x2bcbfd0)

QObject::moveToThread: Current thread (0x2bcbfd0) is not the object's thread (0x30e2ea0).
Cannot move to target thread (0x2bcbfd0)

QObject::moveToThread: Current thread (0x2bcbfd0) is not the object's thread